In [1]:
from mpl_toolkits import mplot3d
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib
import math
import seaborn as sns
import pandas as pd
from sympy import *
from sympy import Symbol, solveset, S, erf, log, sqrt
init_printing(use_unicode=True)

import scipy.optimize as optimize

In [72]:
def U_D(gamma_1,gamma_0,C_1,r,delta):
    return (1-delta)*np.dot(r,gamma_1) - np.dot(np.dot(np.subtract(gamma_1,gamma_0),C_1),np.subtract(gamma_1,gamma_0))

#U_D = term0 * alpha_0 + termA*beta_0^2 + termB&beta_0 + termC

def get_UD_zero_levelset_terms(C_1,r,delta,theta):
    term0 = (1-delta)*r[0]
    termC = ((1-delta)**2/(4*C_1[0,0]) #note: r[0]**2?
             + (1-delta)*r[1]*theta 
             - (C_1[0,1]*(1-delta)*r[0]*theta/C_1[0,0]) #r[1]? Doesn't matter for the r=[1,1] scenario.
             + C_1[0,1]**2 * theta**2 / C_1[0,0]
             - C_1[1,1]*theta**2
            )
    termB = (C_1[0,1]*(1-delta)*r[0] / C_1[0,0]
             - 2*C_1[0,1]**2 * theta / C_1[0,0]
             + 2*C_1[1,1]*theta
            )
    termA = (C_1[0,1]**2 / C_1[0,0]
             - C_1[1,1]
            )
    return term0,termA,termB,termC


def get_UD_zero_levelset_terms(C_1, r, delta, theta):
    # Alpha_0 term coefficient
    alpha_0_term = (1 - delta) * r[0]
    
    # Beta_0^2 term coefficient
    beta_0_squared_term = (C_1[0,1]**2 / C_1[0,0]) - C_1[1,1]
    
    # Beta_0 term coefficient
    beta_0_term = (C_1[0,1] * (1 - delta) * r[0] / C_1[0,0] 
                   - 2 * C_1[0,1]**2 * theta / C_1[0,0] 
                   + 2 * C_1[1,1] * theta)
    
    # Constant term
    constant_term = ((1 - delta)**2 * r[0]**2 / (4 * C_1[0,0]) 
                     + (1 - delta) * r[1] * theta 
                     - (C_1[0,1] * (1 - delta) * r[0] * theta / C_1[0,0]) 
                     + (C_1[0,1]**2 * theta**2 / C_1[0,0]) 
                     - C_1[1,1] * theta**2)
    
    return alpha_0_term, beta_0_squared_term, beta_0_term, constant_term


def get_UD_zero_levelset_terms_updated(C_1, r, delta, theta):
    """
    Returns the updated coefficients for the U_D level set equation in the form:
    alpha_0_term * alpha_0 + beta_0_squared_term * beta_0^2 + beta_0_term * beta_0 + constant_term = 0
    """
    # Alpha_0 term coefficient
    alpha_0_term = (1 - delta) * r[0]

    # Beta_0^2 term coefficient
    beta_0_squared_term = (C_1[0,1]**2 / C_1[0,0]) - C_1[1,1]
    
    
    # Beta_0 term coefficient
    beta_0_term = ((1 - delta) * r[1] 
                   + (1 - delta) * (C_1[0,1] / C_1[0,0]) * r[0] 
                   - 2 * (C_1[1,1] - (C_1[0,1]**2 / C_1[0,0])) * theta)
    
    # Constant term
    constant_term = ((1 - delta) * (-(C_1[0,1] / C_1[0,0]) * r[0] + r[1]) * theta
                     - (C_1[1,1] - (C_1[0,1]**2 / C_1[0,0])) * theta**2
                     + ((1 - delta)**2 * r[0]**2) / (4 * C_1[0,0]))
    
    return alpha_0_term, beta_0_squared_term, beta_0_term, constant_term



def D_strategy(gamma_0,C_1,r,delta,theta):
    if sum(np.equal(gamma_0,[-.1,-.1]))==2:
        return np.array([-.1,-.1]),-.1,"abstain"
    #Abstain condition
    alpha_0 = gamma_0[0]
    beta_0 = gamma_0[1]
    #This is the version without cross-terms
    #abstain_quadratic_A = -C_1[1,1]
    #abstain_quadratic_B = (C_1[0,1]*(1-delta)*r[0]/C_1[0,0]) + 2*theta*C_1[1,1]
    #abstain_quadratic_C = ((1-delta)**2 * r[0]**2 / (4*C_1[0,0])) + (1-delta)*r[1]*theta - (C_1[0,1]*(1-delta)*r[0]*theta/C_1[0,0])-C_1[1,1]*theta**2
    #if 0 > ((1-delta)*r[0])*alpha_0 + abstain_quadratic_A*beta_0**2 + abstain_quadratic_B*beta_0 + abstain_quadratic_C:
    #    return [-.1,-.1],"abstain",-0.1
    term0,termA,termB,termC = get_UD_zero_levelset_terms(C_1,r,delta,theta)
    #print(term0*alpha_0 + termA*beta_0**2 + termB*beta_0 + termC)
    if 0 > term0*alpha_0 + termA*beta_0**2 + termB*beta_0 + termC:
        return [-.1,-.1],-0.1,"abstain"
    
    #Closed-form solution
    gamma_candidate_1 = gamma_0 + 0.5 * (1-delta) * np.dot(np.linalg.inv(C_1),r)
    gamma_candidate_2 = np.array([alpha_0,np.max([beta_0+(1-delta)*r[1]/(2*C_1[1,1]),theta])]) #can likely remove the np.max() and theta option from this
    #Making a slight change here to account for cross-terms:
    gamma_candidate_3 = np.array([np.max([alpha_0,
                                          alpha_0+(1-delta)*r[0]/(2*C_1[0,0]) - (C_1[0,1]/C_1[0,0])*(theta-beta_0)]),
                                  np.max([beta_0,theta])
                                 ])
    
    #Omit unconstrained candidate if it is infeasible
    if gamma_candidate_1[0]<alpha_0 or gamma_candidate_1[1]<np.max([beta_0,theta]):
        candidates = [gamma_candidate_2,gamma_candidate_3]
        strategy_list = ['constrained $\\alpha_1=\\alpha_0$','constrained $\\beta_1=max(\\beta_0,\\theta)$']
    else:
        candidates = [gamma_candidate_1,gamma_candidate_2,gamma_candidate_3]
        strategy_list = ['unconstrained', 'constrained $\\alpha_1=\\alpha_0$','constrained $\\beta_1=max(\\beta_0,\\theta)$']
        
    #print(candidates)
    #Strategy list for explaining remaining candidates
    strategies = [strategy_list[i] for i in range(len(candidates))]
    
    #Evaluate utilities for remaining candidates
    utilities = [U_D(candidates[i],gamma_0,C_1,r,delta) for i in range(len(candidates))]
    index_choice = np.argmax(utilities)

    #if max(utilities)<0:
    #    return np.array([-.1,-.1]),-.1,"abstain"
    
    return candidates[index_choice], utilities[index_choice], strategies[index_choice]
    
def U_G(gamma_0,C_0,C_1,r,delta,theta):
    return delta*np.dot(r,D_strategy(gamma_0,C_1,r,delta,theta)[0]) - np.dot(np.dot(gamma_0,C_0),gamma_0)

def get_candidates(C_0,C_1,delta,r,theta,thetaG,epsilon=0.000001):
    a, b, l = symbols('a b l',real=True)
    C_0_rat = np.array([[Rational(C_0[0,0]),Rational(C_0[0,1])],[Rational(C_0[1,0]),Rational(C_0[1,1])]])
    C_1_rat = np.array([[Rational(C_1[0,0]),Rational(C_1[0,1])],[Rational(C_1[1,0]),Rational(C_1[1,1])]])
    delta_rat = Rational(delta)#0.5
    theta_rat = Rational(theta)
    thetaG_rat = Rational(thetaG)
    r_rat = np.array([Rational(r[0]),Rational(r[1])])
    equation_1 = delta_rat*r_rat[0] - 2*C_0_rat[0,0]*a - 2*C_0_rat[0,1]*b - l*(1-delta_rat)*r_rat[0]
    equation_2 = (delta_rat*C_1_rat[0,1]*r_rat[0]/C_1_rat[0,0]) - 2*C_0_rat[1,1]*b - 2*C_0_rat[0,1]*a - l*(C_1_rat[0,1]*(1-delta_rat)*r_rat[0]/C_1_rat[0,0] - 2*C_1_rat[0,1]**2 * theta_rat/C_1_rat[0,0] + 2*C_1_rat[1,1]*theta_rat + 2*(C_1_rat[0,1]/C_1_rat[0,0]-C_1_rat[1,1])*b)
    term0,termA,termB,termC = get_UD_zero_levelset_terms(C_1_rat,r_rat,delta_rat,theta_rat)
    equation_3 = term0*a + termA*b**2 + termB*b + termC
    raw_results = solve([equation_1,equation_2,equation_3],[a,b,l],domain=S.Reals)#nonlinsolve([equation_1,equation_2,equation_3],[a,b,l],domain=S.Reals)
    raw_results = list(raw_results)
    #print(Float(raw_results,10))
    #print(np.array(raw_results).shape)
    #print(raw_results[0][0])
    results = []
    for i in range(len(raw_results)):
        if isinstance(raw_results[i][0], complex) or isinstance(raw_results[i][1], complex):
            continue
        if ("I" in str(raw_results[i][0]))or("I" in str(raw_results[i][1])):
            continue
        #if N(raw_results[i][0])>1000 or N(raw_results[i][1])>1000:
        #    print("Very large number.")
        #    continue
        results = results + [np.array([N(raw_results[i][j])+epsilon for j in range(2)])]#[np.add(raw_results[i][:2],epsilon)]#[np.array([N(raw_results[i][j])+epsilon for j in range(2)])]
    #results = [np.add(raw_results[i][:2],epsilon) for i in range(len(raw_results))]
    return results

def G_strategy(C_0,C_1,r,delta,theta,thetaG,epsilon=0.000001):
    candidate_1 = 0.5*delta*np.dot(np.linalg.inv(C_0),r)
    candidate_2 = np.array([0,np.max([delta*r[1]/(2*C_0[1,1]),thetaG])])
    candidate_3 = np.array([np.max([0,delta*r[0]/(2*C_0[0,0]) - (C_0[0,1]/C_0[0,0])*thetaG]),thetaG])
    
    term0,termA,termB,termC = get_UD_zero_levelset_terms(C_1,r,delta,theta)
    candidate_5 = [(-1/term0)*(termA*thetaG**2 + termB*thetaG + termC)+epsilon,thetaG+epsilon]
    candidate_6_beta_options = np.real(np.roots([termA,termB,termC]))
    candidate_6_1 = np.array([0+epsilon,candidate_6_beta_options[0]+epsilon])
    candidate_6_2 = np.array([0+epsilon,candidate_6_beta_options[1]+epsilon])
    candidate_7 = np.array([0,thetaG])

    candidate_4_candidates = get_candidates(C_0,C_1,delta,r,theta,thetaG)
    
    candidates_all = [candidate_1, candidate_2,candidate_3,candidate_5,candidate_6_1,candidate_6_2,candidate_7]+candidate_4_candidates
    strategies_all = ['unconstrained','$\\alpha_0=0$','$\\beta_0=\\theta_G$','$U_D=0, \\beta_0=\\theta_G$','$U_D=0, \\alpha_0=0$','$U_D=0, \\alpha_0=0$','$\\alpha_0=0,\\beta_0=\\theta_G$']+["$U_D=0$"]*len(candidate_4_candidates)
    utilities_all = [U_G(g0,C_0,C_1,r,delta,theta) for g0 in candidates_all]
    print(candidates_all)
    print(strategies_all)
    print(utilities_all)
    #Filter infeasible options going back to front
    i = len(candidates_all)-1
    while i>=0:
        #print(strategies_all)
        #for index in np.linspace(len(candidates_all),0):
        #i = int(index)
        #print(i)
        #-0.0001 > term0*alpha_0 + termA*beta_0**2 + termB*beta_0 + termC: 
        if U_D(D_strategy(candidates_all[i],C_1,r,delta,theta)[0],candidates_all[i],C_1,r,delta)< 0:
            candidates_all.pop(i)
            strategies_all.pop(i)
            utilities_all.pop(i)
            #continue
        elif U_G(candidates_all[i],C_0,C_1,r,delta,theta)< 0:
            candidates_all.pop(i)
            strategies_all.pop(i)
            utilities_all.pop(i)
            #continue
        elif (candidates_all[i][0]<0)|(candidates_all[i][1]<thetaG):
            candidates_all.pop(i)
            strategies_all.pop(i)
            utilities_all.pop(i)
            #continue
        i = i-1
    
    if len(candidates_all)<=0:
        return np.array([-.1,-.1]),-.1,"abstain"
    
    choice = np.argmax(utilities_all)
    
    if candidates_all[choice][0]<=0:
        print("Strange case: ")
        print(C_0,C_1,r,delta,theta,thetaG)
        print("Choice: ",candidates_all[choice], utilities_all[choice], strategies_all[choice])
        print("All candidates: ",candidates_all)
        print("All utilities: ",utilities_all)
        print("All strategies: ",strategies_all)
        print("D response and utility: ",D_strategy(candidates_all[i],C_1,r,delta,theta))
    
    return candidates_all[choice], utilities_all[choice], strategies_all[choice]


def G_strategy(C_0, C_1, r, delta, theta_D, theta_G):
    candidate_1 = (delta / 2) * np.linalg.inv(C_0) @ r
    candidate_2 = np.array([0, delta * r[1] / (2 * C_0[1, 1])])
    candidate_3 = np.array([
        delta * r[0] / (2 * C_0[0, 0]) - C_0[0, 1] / C_0[0, 0] * theta_G,
        theta_G
    ])
    candidate_4 = np.array([0, theta_G])

    alpha_0_term, beta_0_squared_term, beta_0_term, constant_term = get_UD_zero_levelset_terms(C_1, r, delta, theta_D)
    beta_0_sym, alpha_0_sym = symbols('beta_0 alpha_0', real=True)
    levelset_eq = Eq(alpha_0_term * r[0] * alpha_0_sym + beta_0_term * beta_0_sym + beta_0_squared_term * beta_0_sym**2 + constant_term, 0)

    beta_solutions = solve(levelset_eq.subs(alpha_0_sym, 0), beta_0_sym)
    candidate_list = [candidate_1, candidate_2, candidate_3, candidate_4]

    for b0 in beta_solutions:
        if b0.is_real:
            b0 = float(b0)
            try:
                alpha_0_solutions = solve(levelset_eq.subs(beta_0_sym, b0), alpha_0_sym)
                if alpha_0_solutions and alpha_0_solutions[0].is_real:
                    a0 = float(alpha_0_solutions[0])
                else:
                    continue
                #a0 = (1/alpha_0_term)*(-(1-delta)*r[1]*b0-(beta_0_squared_term*b0**2+beta_0_term*b0+constant_term))
                #(- (1 - delta) * r[1] * b0 - (beta_0_squared_term * b0**2 + beta_0_term * b0 + constant_term)) / ((1 - delta) * r[0])
                gamma_0 = np.array([a0, b0])
                candidate_list.append(gamma_0)
            except ZeroDivisionError:
                continue

    best_gamma_0 = None
    best_UG = -np.inf
    for gamma_0 in candidate_list:
        if gamma_0 is None or gamma_0[0] < 0 or gamma_0[1] < theta_G:
            continue
        UG = U_G(gamma_0, C_0, C_1, r, delta, theta_D)
        UD = U_D(D_strategy(gamma_0, C_1, r, delta, theta_D)[0], gamma_0, C_1, r, delta)
        if UG >= 0 and UD >= 0 and UG > best_UG:
            best_UG = UG
            best_gamma_0 = gamma_0

    return best_gamma_0, best_UG, 'Using test function'



In [123]:
C_0 = np.array([[1,-0.1],[-0.1,1]])
C_1 = np.array([[1,-0.1],[-0.1,1]])
delta = 0.5 #75
theta = 1#1.5
thetaG = 0
r = np.array([1,1])

gamma_0, u_g0, stratGex = G_strategy(C_0,C_1,r,delta,theta,thetaG)
print()
print("G strategy: ",gamma_0, u_g0, stratGex)
print()
print("D strategy: ",D_strategy(gamma_0,C_1,r,delta,theta))


G strategy:  [0.27777778 0.27777778] 0.6611111111111112 Using test function

D strategy:  (array([0.6, 1. ]), np.float64(0.22111111111111126), 'constrained $\\beta_1=max(\\beta_0,\\theta)$')


In [75]:
#gamma_0 = np.array([50,50])

D_strategy(gamma_0,C_1,r,delta,theta)

(array([0.59, 0.9 ]),
 np.float64(0.29921111111111115),
 'constrained $\\beta_1=max(\\beta_0,\\theta)$')

In [ ]:
D_strategy(gamma_0,C_1,r,delta,theta)

In [151]:
import numpy as np
from sympy import symbols, solve, Eq
from typing import Optional

def D_strategy(gamma_0, C_1, r, delta, theta_D):
    alpha_0, beta_0 = gamma_0

    candidate_1 = gamma_0 + (1 - delta) * (np.linalg.inv(C_1) @ r)
    candidate_2 = np.array([alpha_0, beta_0 + (1 - delta) * r[1] / (2 * C_1[1, 1])])
    candidate_3 = np.array([
        alpha_0 + (1 - delta) * r[0] / (2 * C_1[0, 0]) - (C_1[0, 1] / C_1[0, 0]) * max(0, theta_D - beta_0),
        max(beta_0, theta_D)
    ])
    candidate_4 = np.array([alpha_0, max(beta_0, theta_D)])

    candidates = [candidate_1, candidate_2, candidate_3, candidate_4]

    best_gamma_1 = None
    best_UD = -np.inf

    for gamma_1 in candidates:
        if np.any(gamma_1 < gamma_0) or gamma_1[1] < max(beta_0, theta_D):
            continue
        UD = U_D(gamma_1, gamma_0, C_1, r, delta)
        if UD >= 0 and UD > best_UD:
            best_UD = UD
            best_gamma_1 = gamma_1

    return best_gamma_1

def U_D(gamma_1, gamma_0, C_1, r, delta):
    if gamma_1 is None or gamma_0 is None:
        return -np.inf
    revenue = (1 - delta) * (r @ gamma_1)
    cost = (gamma_1 - gamma_0).T @ C_1 @ (gamma_1 - gamma_0)
    return revenue - cost

def U_G(gamma_0, C_0, C_1, r, delta, theta_D):
    gamma_1 = D_strategy(gamma_0, C_1, r, delta, theta_D)
    if gamma_1 is None or gamma_0 is None:
        return -np.inf
    revenue = delta * (r @ gamma_1)
    cost = gamma_0.T @ C_0 @ gamma_0
    return revenue - cost

def get_UD_zero_levelset_terms(C_1, r, delta, theta):
    # Alpha_0 term coefficient
    alpha_0_term = (1 - delta) * r[0]

    # Beta_0^2 term coefficient
    beta_0_squared_term = (C_1[0,1]**2 / C_1[0,0]) - C_1[1,1]

    # Beta_0 term coefficient
    beta_0_term = (C_1[0,1] * (1 - delta) * r[0] / C_1[0,0] 
                   - 2 * C_1[0,1]**2 * theta / C_1[0,0] 
                   + 2 * C_1[1,1] * theta)

    # Constant term
    constant_term = ((1 - delta)**2 * r[0]**2 / (4 * C_1[0,0]) 
                     + (1 - delta) * r[1] * theta 
                     - (C_1[0,1] * (1 - delta) * r[0] * theta / C_1[0,0]) 
                     + (C_1[0,1]**2 * theta**2 / C_1[0,0]) 
                     - C_1[1,1] * theta**2)

    return alpha_0_term, beta_0_term, beta_0_squared_term, constant_term

def G_strategy(C_0, C_1, r, delta, theta_D, theta_G) -> Optional[np.ndarray]:
    candidate_1 = (delta / 2) * np.linalg.inv(C_0) @ r
    candidate_2 = np.array([0, delta * r[1] / (2 * C_0[1, 1])])
    candidate_3 = np.array([
        delta * r[0] / (2 * C_0[0, 0]) - C_0[0, 1] / C_0[0, 0] * theta_G,
        theta_G
    ])
    candidate_4 = np.array([0, theta_G])

    A, B, C, D = get_UD_zero_levelset_terms(C_1, r, delta, theta_D)
    beta_0_sym, alpha_0_sym = symbols('beta_0 alpha_0', real=True)
    levelset_eq = Eq(A * alpha_0_sym + B * beta_0_sym + C * beta_0_sym**2 + D, 0)

    beta_solutions = solve(levelset_eq.subs(alpha_0_sym, 0), beta_0_sym)
    candidate_list = [candidate_1, candidate_2, candidate_3, candidate_4]

    for b0 in beta_solutions:
        if b0.is_real:
            b0 = float(b0)
            try:
                a0 = (- B * b0 - C * b0**2 - D) / A
                gamma_0 = np.array([a0, b0])
                candidate_list.append(gamma_0)
            except ZeroDivisionError:
                continue

    best_gamma_0 = None
    best_UG = -np.inf
    for gamma_0 in candidate_list:
        if gamma_0 is None or gamma_0[0] < 0 or gamma_0[1] < theta_G:
            continue
        UG = U_G(gamma_0, C_0, C_1, r, delta, theta_D)
        UD = U_D(D_strategy(gamma_0, C_1, r, delta, theta_D), gamma_0, C_1, r, delta)
        if UG >= 0 and UD >= 0 and UG > best_UG:
            best_UG = UG
            best_gamma_0 = gamma_0

    return best_gamma_0, best_UG, 'Using test function'

In [164]:
C_0 = np.array([[1,-0.1],[-0.1,1]])
C_1 = np.array([[1,-0.1],[-0.1,1]])
delta = 0.5 #75
theta = 1.2#1.5
thetaG = 0
r = np.array([1,1])

gamma_0, u_g0, stratGex = G_strategy(C_0,C_1,r,delta,theta,thetaG)
print()
print("G strategy: ",gamma_0, u_g0, stratGex)
print()
print("D strategy: ",D_strategy(gamma_0,C_1,r,delta,theta))


G strategy:  [0.27777778 0.27777778] 0.7711111111111111 Using test function

D strategy:  [0.62 1.2 ]
